# 🎓 EstudeMy — Tarefa de Machine Learning (Supervisionado e Não Supervisionado)

**Projeto:** EstudeMy (Plataforma Gamificada de Ensino)  
**Base de Dados:** 1.000 registros únicos gerados via `Faker('pt_BR')` + `NumPy` refletindo o esquema do MongoDB (`users`, `trilhas`, `fases`, `progresso`, `rankings`).

---

## 1. Respostas das Perguntas da Tarefa

### 📌 Qual é a estrutura do banco de dados?
O banco de dados do projeto é o **MongoDB 7.0 (`estudemy`)**, estruturado em 5 coleções principais:
- **`users`**: `_id`, `nome`, `username`, `email`, `tipoUsuario` (`ALUNO`), `personagem` (`Samurai`, `Mago`, `Guerreiro`), `xpTotal`, `idioma`, `tema`.
- **`trilhas`**: `_id`, `titulo`, `materia` (11 matérias), `dificuldade` (`Facil`, `Medio`, `Dificil`), `popularidade`, `visualizacoes`.
- **`secoes`**: `_id`, `trilhaId`, `titulo`, `ordem`.
- **`fases`**: `_id`, `trilhaId`, `secaoId`, `cenario` (`Vila`, `Montanha`, `Deserto`, `Castelo`), `personagem_selecionado` (`spectre`, `arqueira`, `magoCorrompido`, `cavaleiroDecaido`), `perguntas`.
- **`progresso` / `rankings`**: `userId`, `trilhaId`, `faseId`, `pontuacao`, `acertos`, `erros`, `totalPerguntas`, `mediaAcertos`.

### 📌 O que é o $X$ e o $Y$ no projeto?
- **$X$ (Features / Variáveis de Entrada):** Matriz com **18 características** comportamentais, de gamificação e desempenho do aluno nas fases e trilhas (`idade`, `personagem_cod`, `materia_cod`, `dificuldade_cod`, `cenario_cod`, `inimigo_hp_max`, `trilhas_iniciadas`, `fases_concluidas`, `dias_ofensiva`, `total_perguntas_respondidas`, `total_acertos`, `total_erros`, `media_acertos_pct`, `pontuacao_media`, `hp_medio_restante`, `tempo_medio_fase_seg`, `xp_total`, `nivel_heroi`).
- **$Y$ (Target / Classe Alvo):** Vetor `classe_y` que classifica o nível de domínio e engajamento do aluno em 3 classes:
  - `0` -> **`Basico_Risco_Evasao`** (Baixo rendimento / precisa de reforço e trilhas fáceis)
  - `1` -> **`Intermediario_Regular`** (Progresso consistente / trilhas médias)
  - `2` -> **`Avancado_Mestre`** (Alto domínio e performance nas batalhas / trilhas difíceis)

In [ ]:
import pandas as pd
import numpy as np
from gerar_dados_faker import gerar_1000_registros

# Carrega ou gera os 1.000 registros únicos
df = pd.read_csv('estudemy_dataset_1000.csv')
print(f'Total de registros: {len(df)} | Duplicados: {df.duplicated().sum()}')
df.head(10)

## 2. Definição de $X$ e $Y$ e Pré-processamento

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FEATURES_X = [
    'idade', 'personagem_cod', 'materia_cod', 'dificuldade_cod',
    'cenario_cod', 'inimigo_hp_max', 'trilhas_iniciadas', 'fases_concluidas',
    'dias_ofensiva', 'total_perguntas_respondidas', 'total_acertos',
    'total_erros', 'media_acertos_pct', 'pontuacao_media',
    'hp_medio_restante', 'tempo_medio_fase_seg', 'xp_total', 'nivel_heroi'
]

X = df[FEATURES_X]
y = df['classe_y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=2026, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_all_scaled = scaler.fit_transform(X)

print('Formato de X_train:', X_train.shape, '| Formato de X_test:', X_test.shape)

## 3. Aprendizado Supervisionado ($X$ e $Y$): Árvore de Decisão, Random Forest e Rede Neural

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nomes_classes = ['0: Basico_Risco_Evasao', '1: Intermediario_Regular', '2: Avancado_Mestre']

# 1. Árvore de Decisão
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=8, random_state=2026)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print('=== 1. ÁRVORE DE DECISÃO ===')
print(f'Acurácia: {accuracy_score(y_test, y_pred_dt)*100:.2f}%')
print(classification_report(y_test, y_pred_dt, target_names=nomes_classes))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=150, max_depth=8, min_samples_leaf=4, random_state=2026)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('=== 2. RANDOM FOREST ===')
print(f'Acurácia: {accuracy_score(y_test, y_pred_rf)*100:.2f}%')
print(classification_report(y_test, y_pred_rf, target_names=nomes_classes))

# 3. Rede Neural (MLPClassifier)
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', max_iter=500, random_state=2026)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)
print('=== 3. REDE NEURAL (MLP) ===')
print(f'Acurácia: {accuracy_score(y_test, y_pred_mlp)*100:.2f}%')
print(classification_report(y_test, y_pred_mlp, target_names=nomes_classes))

## 4. Aprendizado Não Supervisionado (Apenas $X$ $\rightarrow$ Cluster e Cruzamento com a Classe $Y$)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

kmeans = KMeans(n_clusters=3, random_state=2026, n_init=30)
clusters_brutos = kmeans.fit_predict(X_all_scaled)

# Alinha a numeração dos clusters por XP crescente (0=Iniciante, 1=Intermediário, 2=Veterano)
df['cluster_bruto'] = clusters_brutos
ordem = df.groupby('cluster_bruto')['xp_total'].mean().sort_values().index.tolist()
df['cluster_kmeans'] = df['cluster_bruto'].map({old: new for new, old in enumerate(ordem)})

print(f'Silhouette Score (k=3): {silhouette_score(X_all_scaled, df["cluster_kmeans"]):.4f}')
print(f'Adjusted Rand Index (Cluster vs Classe Y): {adjusted_rand_score(y, df["cluster_kmeans"]):.4f}')
print(f'Correspondência Direta Cluster x Classe: {accuracy_score(y, df["cluster_kmeans"])*100:.2f}%\n')

pd.crosstab(df['cluster_kmeans'], df['classe_nome'], margins=True)